# Featurizing a small Target-2 batch

This notebook runs the main utilities of `morphology_featurization_tools` on a
few wells of one JUMP Cell Painting Target-2 plate:

1. download a small batch of fields from the Target-2 plates;
2. look at one field and at how the batch is organized;
3. compute CellProfiler-style well profiles, with per-plate illumination
   correction and Cellpose segmentation;
4. compute OpenPhenom autoencoder well profiles from the same fields.

It needs the environment described in the README and internet access.
Section 4 also needs the optional OpenPhenom model code: run
`bash install_openphenom.sh` from the repository root, which asks you to
accept OpenPhenom's license first. The first run downloads the JUMP metadata
tables and image index, the Cellpose model and the OpenPhenom weights, which
are cached for later runs. Cellpose and OpenPhenom use a GPU when one is
available and are much slower without one.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from morphology_featurization_tools import (
    FeatureAggregator,
    FeatureOrchestrator,
    FieldMaskSegmenter,
    IlluminationCorrection,
    JUMPCellPaintingDataset,
    JUMPTarget2Dataset,
    build_feature,
)
from morphology_featurization_tools.datasets.jump_cell_painting import control_ids

WELL_KEYS = ["Metadata_Source", "Metadata_Plate", "Metadata_Well"]

N_CONTROL_WELLS = 2    # negative-control wells in the batch
N_COMPOUND_WELLS = 2   # compound wells in the batch
N_JOBS = 4             # worker processes for the CellProfiler-style features
GPU = torch.cuda.is_available()
print("GPU available:", GPU)

GPU available: True


## 1. Download a small batch

`JUMPTarget2Dataset` builds a *site table* for every well of the Target-2
plates: one row per site (field of view) with its metadata and one image URL
per channel. Source 9 is left out by default. Images are fetched only when an
item is requested.

In [ ]:
target2 = JUMPTarget2Dataset()
# sites = target2.site_table

# n_wells = len(sites[WELL_KEYS].drop_duplicates())
# print(f"{len(sites):,} sites in {n_wells:,} wells on {sites['Metadata_Plate'].nunique()} plates "
#       f"from {sites['Metadata_Source'].nunique()} sources")
# print("channels:", target2.channels)
# sites.head()

`JUMPCellPaintingDataset` accepts any subset of a site table. The batch here is
a few negative-control (DMSO) wells and a few compound wells from one plate,
with every site of each well. The negative controls are read from JUMP's
`perturbation_control` table.

In [ ]:
plate = sorted(sites["Metadata_Plate"].unique())[0]
plate_sites = sites[sites["Metadata_Plate"] == plate]

wells = plate_sites[WELL_KEYS + ["Metadata_JCP2022"]].drop_duplicates()
is_control = wells["Metadata_JCP2022"].isin(control_ids("negcon", "compound"))
chosen = pd.concat([wells[is_control].head(N_CONTROL_WELLS),
                    wells[~is_control].head(N_COMPOUND_WELLS)], ignore_index=True)

batch = JUMPCellPaintingDataset(plate_sites.merge(chosen[WELL_KEYS], on=WELL_KEYS))
print(f"plate {plate}: {len(chosen)} wells, {len(batch)} sites")

In [ ]:
# Sites are fetched from the Cell Painting Gallery's public S3 bucket, several
# at a time by the DataLoader's workers. collate_fn=list keeps each item a dict.
loader = DataLoader(batch, batch_size=4, num_workers=4, collate_fn=list)
samples = [item for items in loader for item in items]
print(f"downloaded {len(samples)} sites")

## 2. Structure of the batch

Each item is a dict holding `image`, a `(channels, height, width)` array with
the channels in the order of `batch.channels`, and the site's metadata. The
batch has these wells and sites:

In [ ]:
batch.site_table.groupby(WELL_KEYS + ["Metadata_JCP2022"]).size().rename("sites").to_frame()

In [ ]:
sample = samples[0]
print(pd.Series({key: value for key, value in sample.items() if key != "image"}).to_string())
print("image:", sample["image"].shape, sample["image"].dtype)

fig, axes = plt.subplots(1, len(batch.channels), figsize=(4 * len(batch.channels), 4.6))
for ax, name, plane in zip(axes, batch.channels, sample["image"]):
    low, high = np.percentile(plane, [1, 99.5])   # display contrast only
    ax.imshow(plane, cmap="gray", vmin=low, vmax=high)
    ax.set_title(name)
    ax.axis("off")
fig.suptitle(f"Plate {sample['Metadata_Plate']}, well {sample['Metadata_Well']}, "
             f"site {sample['Metadata_Site']} ({sample['Metadata_JCP2022']})")
fig.tight_layout()

## 3. CellProfiler-style features

1. **Illumination correction.** For each plate and channel, the mean field is
   smoothed with a wide Gaussian (σ = 100 px) into an illumination function,
   and every field is divided by it.
2. **Segmentation.** Cellpose segments cells from the DNA and AGP channels.
3. **Measurement.** cp_measure computes the CellProfiler feature families for
   every cell. `FeatureOrchestrator` runs the extractors and aggregates the
   cells of each well by their median.

The illumination functions are fitted here on the batch's own fields. A
profiling run fits them on every field of the plate, which needs the whole
plate downloaded. `fit` accepts a generator, so the fields can be streamed:

```python
plate_loader = DataLoader(JUMPCellPaintingDataset(plate_sites), batch_size=4,
                          num_workers=4, collate_fn=list)
illumination.fit((item["image"] for items in plate_loader for item in items),
                 [plate] * len(plate_sites))
```

In [ ]:
illumination = IlluminationCorrection(smoothing_sigma=100.0, epsilon=1e-6)
plates = [s["Metadata_Plate"] for s in samples]
illumination.fit((s["image"] for s in samples), plates)

corrected_images = illumination.transform([s["image"] for s in samples], plates)
corrected = [{**s, "image": image} for s, image in zip(samples, corrected_images)]

In [ ]:
channels = list(batch.channels)

# Mask-derived families are measured once per cell; intensity-derived families
# once per cell and channel.
cellprofiler_families = {
    "cp_sizeshape": None,
    "cp_zernike": None,
    "cp_feret": None,
    "cp_intensity": channels,
    "cp_texture": channels,
    "cp_granularity": channels,
    "cp_radial_distribution": channels,
}
colocalization_pairs = [("DNA", "RNA"), ("ER", "AGP")]

extractors = []
for family, family_channels in cellprofiler_families.items():
    if family_channels is None:
        extractors.append(build_feature(family, channel=0))
        continue
    for name in family_channels:
        extractors.append(build_feature(family, channel=channels.index(name),
                                        channel_name=name))
for first, second in colocalization_pairs:
    extractors.append(build_feature(
        "cp_colocalization",
        channel_a=channels.index(first), channel_b=channels.index(second),
        channel_a_name=first, channel_b_name=second,
    ))

segmenter = FieldMaskSegmenter(
    canvas_size=256,
    cellpose_channels=[channels.index("DNA"), channels.index("AGP"), None],
    diameter=None,   # estimated by Cellpose
    gpu=GPU,
)
orchestrator = FeatureOrchestrator(
    extractors=extractors,
    grouping_keys=WELL_KEYS,
    field_mask_segmenter=segmenter,
    n_jobs=N_JOBS,
)

In [ ]:
engineered = orchestrator.run(corrected)
print(f"{len(engineered)} wells x {engineered.shape[1] - len(WELL_KEYS)} features")
engineered.head()

## 4. Autoencoder features

OpenPhenom (Kraus et al. 2024) is a masked autoencoder trained on Cell Painting
images. It receives the fields **without** illumination correction. Each field
is cut into 256 × 256 patches, every patch is encoded, and the patch embeddings
are averaged into one 384-dimensional vector per field; the fields of a well
are then averaged into the well's profile.

This section needs the OpenPhenom model code, installed with
`bash install_openphenom.sh` (see the README).

In [ ]:
# OpenPhenom receives the uncorrected fields, with the channels in this order.
autoencoder_channels = ["DNA", "RNA", "ER", "AGP", "Mito"]
order = [channels.index(name) for name in autoencoder_channels]
autoencoder_fields = [{**s, "image": s["image"][order]} for s in samples]

autoencoder = build_feature("autoencoder", channel_map=[0, 1, 2, 3, 4], batch_size=8)
field_embeddings, field_metadata = autoencoder.extract_fields_from_dataset(autoencoder_fields)
print("field embeddings:", field_embeddings.shape)

well_mean = FeatureAggregator(grouping_keys=WELL_KEYS,
                              aggregator_fn=lambda rows: rows.mean(axis=0))
well_values, well_labels = well_mean.aggregate(np.asarray(field_embeddings), field_metadata)
autoencoder_profiles = pd.concat(
    [well_labels.reset_index(drop=True),
     pd.DataFrame(well_values, columns=autoencoder.feature_names())],
    axis=1,
)
autoencoder_profiles.head()

Both blocks are keyed by well, so they join into one profile per well:

In [ ]:
profiles = (chosen
            .merge(engineered, on=WELL_KEYS, how="left")
            .merge(autoencoder_profiles, on=WELL_KEYS, how="left"))
print(profiles.shape)
profiles.iloc[:, :8]

## Next steps

- `JUMPCompoundDataset`, `JUMPPositiveControlDataset` and `JUMPCRISPRDataset`
  build site tables for chosen compounds or genes and their negative controls.
- `available_features()` lists every registered feature family.
- `TypicalVariationNormalization` and `MADRobustize` normalize well profiles
  against the negative-control wells.